In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import pypsa
import xlsxwriter
import tz_pypsa
import tz_pypsa.wrangle as wrangle
import pandas as pd
# import tz_solve
import plotly.express as px
import plotly.graph_objects as go
from tz_pypsa.model import Model
from tz_pypsa.utils import get_examples
import os
import glob

In [2]:
n = pypsa.Network()
n.import_from_netcdf("C:/Users/jy/TransitionZero/Google - CFE - Documents/04. Country Specific Vault/Japan/02. Data & Results/Outputs/1_Diagnosis/TP1/Run002/JPN_P1_JPN08/solved_networks/hourly_matching_CFE100_2030.nc")

INFO:pypsa.io:Imported network hourly_matching_CFE100_2030.nc has buses, carriers, generators, links, loads, storage_units


In [12]:
def get_ci_cost_summary(n : pypsa.Network) -> pd.DataFrame:
    '''Returns a summary of the costs for C&I generators, storage units and links
    '''
    ci_generator_costs = (
        n.generators.loc[
            n.generators.index.str.contains('C&I')
        ]
        [['carrier','p_nom','p_nom_opt','capital_cost','marginal_cost']]
        #.reset_index()
    )

    ci_generator_p_max_pu = (
        n.generators_t.p_max_pu.transpose().loc[
            n.generators_t.p_max_pu.transpose().index.str.contains('C&I')
        ]
        .transpose()
        # [['p_max_pu']]
        #.reset_index()
    )

    ci_generator_costs['dispatch'] = n.generators_t.p[ ci_generator_costs.index ].sum()
    ci_generator_costs['potential_dispatch'] = (
        ci_generator_costs.p_nom_opt[ ci_generator_costs.index ] 
        * ci_generator_p_max_pu[ ci_generator_costs.index ] 
        ).sum()
    ci_generator_costs['curtailment'] = ci_generator_costs['potential_dispatch'] - ci_generator_costs['dispatch']
    ci_generator_costs['curtailment_perc'] = ci_generator_costs['curtailment']/ci_generator_costs['potential_dispatch']

    # storage
    ci_storage_costs = (
        n.storage_units.loc[
            n.storage_units.index.str.contains('C&I')
        ]
        [['carrier','p_nom','p_nom_opt','capital_cost','marginal_cost']]
        #.reset_index()
    )

    ci_storage_costs['dispatch'] = n.storage_units_t.p_dispatch[ ci_storage_costs.index ].sum()

    # links
    ci_links_costs = (
        n.links.loc[
            n.links.index.str.contains('C&I')
        ]
        [['carrier','p_nom','p_nom_opt','capital_cost','marginal_cost']]
        #.reset_index()
    )

    # zero link costs because they are virtual
    ci_links_costs['capital_cost'] = 0
    ci_links_costs['marginal_cost'] = 0

    ci_links_costs['dispatch'] = n.links_t.p0[ ci_links_costs.index ].sum()

    df = pd.concat([ci_generator_costs, ci_storage_costs, ci_links_costs]).round(3)

    df.loc[:, 'capex'] = df['p_nom_opt'] * df['capital_cost']
    df.loc[:, 'opex'] = df['dispatch'] * df['marginal_cost']

    # calculate import costs
    import_links_t = n.links_t.p0.filter(regex='C&I').filter(regex='Import').sum(axis=1)
    import_link_p = n.buses_t.marginal_price.filter(regex='^(?!.*C&I)').mean(axis=1)
    import_cost = ( import_links_t * import_link_p ).sum() 

    # append to df
    df.loc[ df.index.str.contains('Import'), 'import_cost' ] = import_cost

    # calculate export revenues
    export_links_t = n.links_t.p0.filter(regex='C&I').filter(regex='Export').sum(axis=1)
    export_link_p = n.buses_t.marginal_price.filter(regex='^(?!.*C&I)').mean(axis=1)
    export_revenue = -( export_links_t * export_link_p ).sum().sum()

    # append to df
    df.loc[ df.index.str.contains('Export'), 'export_revenue' ] = export_revenue

    # fillna
    df.fillna(0, inplace=True)

    df.loc[:, 'unit_cost'] = (df['capex'] + df['opex'] + df['import_cost'] + df['export_revenue']) / df['dispatch']

    return df

In [ ]:
# Get both individual prices and average grid price in one DataFrame
prices = (
    n
    .buses_t
    .marginal_price
    .filter(regex='^(?!.*C&I)')
    .reset_index()
).assign(grid_price = lambda df: df.iloc[:, 1:].mean(axis=1))  # Calculate mean of all price columns except 'snapshot'

# Initialize the DataFrame with all C&I related data
ci_data = pd.DataFrame(index=n.loads_t.p.index)

# Load
ci_data['C&I Load'] = n.loads_t.p.filter(regex='C&I').sum(axis=1)

# Import/Export volumes
import_volume = n.links_t.p0.filter(regex='C&I').filter(regex='Import')
export_volume = n.links_t.p0.filter(regex='C&I').filter(regex='Export')

# Calculate costs while maintaining DataFrame structure
# Use the grid_price column from prices DataFrame
import_costs = import_volume.mul(prices['grid_price'], axis=0)
export_revenue = -export_volume.mul(prices['grid_price'], axis=0)

# Store both detailed and total values
ci_data['C&I Import Volume'] = import_volume.sum(axis=1)
ci_data['C&I Export Volume'] = export_volume.sum(axis=1)

# Store individual import/export costs by link
for col in import_costs.columns:
    link_name = col.replace('Import', 'Import Cost')
    ci_data[link_name] = import_costs[col]

for col in export_revenue.columns:
    link_name = col.replace('Export', 'Export Revenue')
    ci_data[link_name] = export_revenue[col]

# Store totals
ci_data['C&I Import Cost Total'] = import_costs.sum(axis=1)
ci_data['C&I Export Revenue Total'] = export_revenue.sum(axis=1)


In [ ]:
# Get average grid price as a DataFrame
grid_price = (
    n
    .buses_t
    .marginal_price
    .filter(regex='^(?!.*C&I)')
    .mean(axis=1)
    .to_frame(name='grid_price')  # Convert to DataFrame with column name
)

# Initialize the DataFrame with all C&I related data
ci_data = pd.DataFrame(index=n.loads_t.p.index)

# Load
ci_data['C&I Load'] = n.loads_t.p.filter(regex='C&I').sum(axis=1)

# Import/Export volumes
import_volume = n.links_t.p0.filter(regex='C&I').filter(regex='Import')
export_volume = n.links_t.p0.filter(regex='C&I').filter(regex='Export')

# Calculate costs while maintaining DataFrame structure
# Now grid_price is a DataFrame, we need to use its values for multiplication
import_costs = import_volume.mul(grid_price['grid_price'], axis=0)
export_revenue = -export_volume.mul(grid_price['grid_price'], axis=0)

# Store both detailed and total values
ci_data['C&I Import Volume'] = import_volume.sum(axis=1)
ci_data['C&I Export Volume'] = export_volume.sum(axis=1)

# Store individual import/export costs by link
for col in import_costs.columns:
    link_name = col.replace('Import', 'Import Cost')
    ci_data[link_name] = import_costs[col]

for col in export_revenue.columns:
    link_name = col.replace('Export', 'Export Revenue')
    ci_data[link_name] = export_revenue[col]

# Store totals
ci_data['C&I Import Cost Total'] = import_costs.sum(axis=1)
ci_data['C&I Export Revenue Total'] = export_revenue.sum(axis=1)


In [ ]:
# Get average grid price (calculated once to avoid repetition)
grid_price = n.buses_t.marginal_price.filter(regex='^(?!.*C&I)').mean(axis=1)

# Initialize the DataFrame with all C&I related data
ci_data = pd.DataFrame(index=n.loads_t.p.index)

# Load
ci_data['C&I Load'] = n.loads_t.p.filter(regex='C&I').sum(axis=1)

# Import/Export volumes
import_volume = n.links_t.p0.filter(regex='C&I').filter(regex='Import')
export_volume = n.links_t.p0.filter(regex='C&I').filter(regex='Export')

# Calculate costs while maintaining DataFrame structure
import_costs = import_volume.mul(grid_price, axis=0)  # Element-wise multiplication with broadcasting
export_revenue = -export_volume.mul(grid_price, axis=0)  # Negative because exports generate revenue

# Store both detailed and total values
ci_data['C&I Import Volume'] = import_volume.sum(axis=1)
ci_data['C&I Export Volume'] = export_volume.sum(axis=1)

# Store individual import/export costs by link
for col in import_costs.columns:
    link_name = col.replace('Import', 'Import Cost')
    ci_data[link_name] = import_costs[col]

for col in export_revenue.columns:
    link_name = col.replace('Export', 'Export Revenue')
    ci_data[link_name] = export_revenue[col]

# Store totals
ci_data['C&I Import Cost Total'] = import_costs.sum(axis=1)
ci_data['C&I Export Revenue Total'] = export_revenue.sum(axis=1)


In [13]:
get_ci_cost_summary(n)

,carrier,p_nom,p_nom_opt,capital_cost,marginal_cost,dispatch,potential_dispatch,curtailment,curtailment_perc,capex,opex,import_cost,export_revenue,unit_cost
JPN08 C&I Grid-solar-unspecified-ext-2030-PPA,SolarGrid,0.00,695.237,178613.000,0.0,787192.215,1087389.866,300197.651,0.276,1.241784e+08,0.0,0.000000,0.000000e+00,157.748468
JPN08 C&I Grid-onshorewind-unspecified-ext-2030-PPA,OnshoreWind,0.00,281.946,351782.000,0.0,570076.245,681713.783,111637.538,0.164,9.918353e+07,0.0,0.000000,0.000000e+00,173.982917
JPN08 C&I Grid-Batteries,Batteries,0.00,697.233,290278.849,0.0,946847.679,0.000,0.000,0.000,2.023920e+08,0.0,0.000000,0.000000e+00,213.753487
JPN08 C&I Grid Imports,AC,204.28,5145.818,0.000,0.0,0.246,0.000,0.000,0.000,0.000000e+00,0.0,14.152558,0.000000e+00,57.530725
JPN08 C&I Grid Exports,AC,0.00,5160.037,0.000,0.0,221330.702,0.000,0.000,0.000,0.000000e+00,0.0,0.000000,-1.370952e+07,-61.941322
JPN08 C&I Storage Charge,AC,0.00,5162.268,0.000,0.0,253108.207,0.000,0.000,0.000,0.000000e+00,0.0,0.000000,0.000000e+00,0.000000
JPN08 C&I Storage Discharge,AC,0.00,5163.541,0.000,0.0,223824.259,0.000,0.000,0.000,0.000000e+00,0.0,0.000000,0.000000e+00,0.000000


In [18]:
# Get average grid price (calculated once to avoid repetition)
grid_price = n.buses_t.marginal_price.filter(regex='^(?!.*C&I)').mean(axis=1)

# Initialize the DataFrame with all C&I related data
ci_data = pd.DataFrame(index=n.loads_t.p.index)

# Load
ci_data['C&I Load'] = n.loads_t.p.filter(regex='C&I').sum(axis=1)

# C&I Price
ci_data['C&I Price'] = grid_price

# Import/Export volumes and costs
ci_data['C&I Import Volume'] = 
ci_data['C&I Export Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Export').sum(n.links_t.p0.filter(regex='C&I').filter(regex='Import').sum(axis=1)axis=1)
ci_data['C&I Import Cost'] = ci_data['C&I Import Volume'] * grid_price
ci_data['C&I Export Revenue'] = -ci_data['C&I Export Volume'] * grid_price  # Negative because exports generate revenue

In [64]:
prices = (
    n
    .buses_t
    .marginal_price
    .filter(regex='^(?!.*C&I)')
    .assign(ci_grid_price = lambda df: df.iloc[:, 1:].mean(axis=1))  # Calculate mean of all price columns except 'snapshot'
    .reset_index()
)

In [63]:
n.buses_t.marginal_price.filter(regex='(?!.*C&I)')

Bus,JPN01,JPN02,JPN03,JPN04,JPN05,JPN06,JPN07,JPN08,JPN08 C&I Grid,JPN08 C&I Storage,JPN09
snapshot,,,,,,,,,,,
2030-01-01 00:00:00,61.391291,63.289965,63.29,63.29,63.290002,63.290000,61.391300,61.391298,-0.000000,0.009910,60.803237
2030-01-01 01:00:00,61.391310,63.289966,63.29,63.29,63.290001,63.290000,61.391300,61.391298,-0.000000,0.009909,60.803232
2030-01-01 02:00:00,61.391596,63.289969,63.29,63.29,63.290002,63.290000,61.391300,61.391298,-263116.128141,0.009909,60.803235
2030-01-01 03:00:00,61.391502,63.289970,63.29,63.29,63.290001,63.290000,61.391300,61.391298,-0.000000,0.009906,60.803233
2030-01-01 04:00:00,61.391338,63.289970,63.29,63.29,63.290000,63.289999,61.391300,61.391298,-263116.128495,0.009903,60.803231
...,...,...,...,...,...,...,...,...,...,...,...
2030-12-31 19:00:00,59.549582,61.391302,63.29,63.29,63.289991,63.290000,61.391300,61.391298,-263116.128600,0.009908,60.803229
2030-12-31 20:00:00,59.549582,61.391301,63.29,63.29,63.289991,63.290000,61.391300,61.391298,-263116.128601,0.009909,60.803229
2030-12-31 21:00:00,59.549582,61.391301,63.29,63.29,63.289991,63.289999,61.391300,61.391298,-0.000000,0.009908,60.803229


In [55]:
grid_price = n.buses_t.marginal_price.filter(regex='^(?!.*C&I)').mean(axis=1)

In [56]:
grid_price.to_dataframe(columns=['snapshot', 'Value'])

snapshot
2030-01-01 00:00:00    62.380788
2030-01-01 01:00:00    62.380790
2030-01-01 02:00:00    62.380822
2030-01-01 03:00:00    62.380812
2030-01-01 04:00:00    62.380793
                         ...    
2030-12-31 19:00:00    61.965189
2030-12-31 20:00:00    61.965189
2030-12-31 21:00:00    61.965189
2030-12-31 22:00:00    61.965189
2030-12-31 23:00:00    61.965189
Length: 8760, dtype: float64

In [49]:
interconnector_lookup = (n
                            .links
                            .reset_index()[['Link', 'bus0', 'bus1']]
                            .rename(
                                columns={'bus0': 'Node', 'bus1': 'Node_Destination'})
                        )


# Get average grid price
grid_price = (
    n
    .buses_t
    .marginal_price
    .filter(regex='^(?!.*C&I)')
    .mean(axis=1)
                )

# Import/Export volumes and costs
import_cost = (
    n
    .links_t
    .p0
    .filter(regex='C&I')
    .filter(regex='Import')
    .mul(grid_price, axis=0)
    ).reset_index()

import_cost = pd.melt(
    import_cost,
    id_vars='snapshot',
    var_name='Link', 
    value_name='Value'
)

if import_cost is not None:
    try:
        import_cost = pd.merge(
            import_cost,
            interconnector_lookup,
            on='Link',
            how='left'
        ).rename(
            columns={'Node': 'Node_Destination', 'Node_Destination': 'Node'}
        )

    except Exception as e:
        print(f"Skipping import_cost: {e}")
        import_cost = None

In [51]:
grid_price


snapshot
2030-01-01 00:00:00    62.380788
2030-01-01 01:00:00    62.380790
2030-01-01 02:00:00    62.380822
2030-01-01 03:00:00    62.380812
2030-01-01 04:00:00    62.380793
                         ...    
2030-12-31 19:00:00    61.965189
2030-12-31 20:00:00    61.965189
2030-12-31 21:00:00    61.965189
2030-12-31 22:00:00    61.965189
2030-12-31 23:00:00    61.965189
Length: 8760, dtype: float64

In [47]:
if import_cost is not None:
    try:
        import_cost = pd.merge(
            import_cost,
            interconnector_lookup,
            on='Link',
            how='left'
        ).rename(
            columns={'Node': 'Node_Destination', 'Node_Destination': 'Node'}
        )

    except Exception as e:
        print(f"Skipping import_cost: {e}")
        import_cost = None

Skipping import_cost: 'Link'


In [16]:
# Get average grid price (calculated once to avoid repetition)
grid_price = n.buses_t.marginal_price.filter(regex='^(?!.*C&I)').mean(axis=1)

# Initialize the DataFrame with all C&I related data
ci_data = pd.DataFrame(index=n.loads_t.p.index)

# Load
ci_data['C&I Load'] = n.loads_t.p.filter(regex='C&I').sum(axis=1)

# C&I Price
ci_data['C&I Price'] = grid_price

# Import/Export volumes and costs
ci_data['C&I Import Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Import').sum(axis=1)
ci_data['C&I Export Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Export').sum(axis=1)
ci_data['C&I Import Cost'] = ci_data['C&I Import Volume'] * grid_price
ci_data['C&I Export Revenue'] = -ci_data['C&I Export Volume'] * grid_price  # Negative because exports generate revenue

# Group generators by technology type
ci_generators = n.generators[n.generators.index.str.contains('C&I')]
tech_groups = ci_generators.groupby('carrier')

# Generation and Curtailment by technology
for tech, group in tech_groups:
    tech_name = f'C&I {tech}'

    # Generation
    ci_data[f'{tech_name} Generation'] = n.generators_t.p[group.index].sum(axis=1)

    # Potential Generation
    ci_data[f'{tech_name} Potential'] = (
        n.generators_t.p_max_pu[group.index] * 
        n.generators.p_nom_opt[group.index]
    ).sum(axis=1)

    # Curtailment
    ci_data[f'{tech_name} Curtailment'] = (
        ci_data[f'{tech_name} Potential'] - 
        ci_data[f'{tech_name} Generation']
    )

    # Hourly expanded CAPEX
    yearly_capex = (
        n.generators.capital_cost[group.index] * 
        n.generators.p_nom_opt[group.index]
    ).sum()

    capex_hourly = yearly_capex / 8760
    ci_data[f'{tech_name} CAPEX'] = pd.Series([capex_hourly] * len(ci_data.index), index=ci_data.index)

    # OPEX
    ci_data[f'{tech_name} OPEX'] = (
        n.generators_t.p[group.index] * 
        group['marginal_cost'].values
    ).sum(axis=1)

    # Total cost
    ci_data[f'{tech_name} Total Cost'] = (
        ci_data[f'{tech_name} CAPEX'] + 
        ci_data[f'{tech_name} OPEX']
    )

# Sum up CAPEX and OPEX across all technologies
capex_cols = [col for col in ci_data.columns if 'CAPEX' in col]
opex_cols = [col for col in ci_data.columns if 'OPEX' in col]
curtailment_cols = [col for col in ci_data.columns if 'Curtailment' in col]

ci_data['C&I Total CAPEX'] = ci_data[capex_cols].sum(axis=1)
ci_data['C&I Total OPEX'] = ci_data[opex_cols].sum(axis=1)
ci_data['C&I Total Curtailment'] = ci_data[curtailment_cols].sum(axis=1)

# Calculate total system cost
ci_data['C&I Total Cost'] = (
    ci_data['C&I Total CAPEX'] + 
    ci_data['C&I Total OPEX'] + 
    ci_data['C&I Import Cost'] + 
    ci_data['C&I Export Revenue']
)

# Calculate system-wide unit costs
ci_data['Unit Cost (All Energy)'] = ci_data['C&I Total Cost']/(
    ci_data['C&I Load'] + 
    ci_data['C&I Export Volume'] + 
    ci_data['C&I Total Curtailment']
)

ci_data['Unit Cost (C&I Energy only)'] = ci_data['C&I Total Cost']/ci_data['C&I Load']

# Looping to calculate unit cost breakdown
for tech, group in tech_groups:
    tech_name = f'C&I {tech}'
    ci_data[f'{tech_name} Unit Cost'] = ci_data[f'{tech_name} Total Cost']/(
        ci_data['C&I Load'] + 
        ci_data['C&I Export Volume'] + 
        ci_data['C&I Total Curtailment']
        )

In [17]:
ci_data

,C&I Load,C&I Price,C&I Import Volume,C&I Export Volume,C&I Import Cost,C&I Export Revenue,C&I OnshoreWind Generation,C&I OnshoreWind Potential,C&I OnshoreWind Curtailment,C&I OnshoreWind CAPEX,...,C&I SolarGrid OPEX,C&I SolarGrid Total Cost,C&I Total CAPEX,C&I Total OPEX,C&I Total Curtailment,C&I Total Cost,Unit Cost (All Energy),Unit Cost (C&I Energy only),C&I OnshoreWind Unit Cost,C&I SolarGrid Unit Cost
snapshot,,,,,,,,,,,,,,,,,,,,,
2030-01-01 00:00:00,135.48,62.380788,1.502909e-07,3.397952,0.000009,-211.966954,141.673364,152.532897,10.859533,11322.328779,...,0.0,14175.616038,25497.944817,0.0,10.859533,25285.977872,168.868722,186.639931,75.614525,94.669788
2030-01-01 01:00:00,135.08,62.380790,1.505150e-07,3.361920,0.000009,-209.719228,141.271696,152.250951,10.979255,11322.328779,...,0.0,14175.616038,25497.944817,0.0,10.979255,25288.225598,169.241244,187.209251,75.774593,94.870195
2030-01-01 02:00:00,137.56,62.380822,1.508912e-07,2.643384,0.000009,-164.896444,142.511360,150.841220,8.329860,11322.328779,...,0.0,14175.616038,25497.944817,0.0,8.329860,25333.048382,170.554737,184.159991,76.227574,95.437329
2030-01-01 03:00:00,138.00,62.380812,1.514246e-07,3.388815,0.000009,-211.397040,144.273570,155.634305,11.360735,11322.328779,...,0.0,14175.616038,25497.944817,0.0,11.360735,25286.547786,165.542535,183.235854,74.123483,92.802997
2030-01-01 04:00:00,133.80,62.380793,1.520873e-07,5.477352,0.000009,-341.681537,143.653096,160.991283,17.338187,11322.328779,...,0.0,14175.616038,25497.944817,0.0,17.338187,25156.263289,160.624313,188.013926,72.293777,90.512194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2030-12-31 19:00:00,129.64,61.965189,1.650526e-07,7.473269,0.000010,-463.082498,142.610081,162.682961,20.072879,11322.328779,...,0.0,14175.616038,25497.944817,0.0,20.072879,25034.862329,159.268884,193.110632,72.031339,90.183621
2030-12-31 20:00:00,126.72,61.965189,1.589420e-07,7.392234,0.000010,-458.061188,139.545444,159.581552,20.036108,11322.328779,...,0.0,14175.616038,25497.944817,0.0,20.036108,25039.883639,162.440175,197.600092,73.450863,91.960872
2030-12-31 21:00:00,122.52,61.965189,1.590419e-07,10.819507,0.000010,-670.432786,139.568569,162.682961,23.114391,11322.328779,...,0.0,14175.616038,25497.944817,0.0,23.114391,24827.512040,158.688996,202.640484,72.368467,90.605707


In [4]:
ci_data

,C&I Load,C&I Price,C&I Import Volume,C&I Export Volume,C&I Import Cost,C&I Export Revenue,C&I OnshoreWind Generation,C&I OnshoreWind Potential,C&I OnshoreWind Curtailment,C&I OnshoreWind CAPEX,...,C&I SolarGrid CAPEX,C&I SolarGrid OPEX,C&I SolarGrid Total Cost,C&I Total CAPEX,C&I Total OPEX,C&I Total Curtailment,C&I Total Cost,Unit Cost (All Energy),Unit Cost (C&I Energy only),solar_unit_cost
snapshot,,,,,,,,,,,,,,,,,,,,,
2030-01-01 00:00:00,135.48,62.380788,1.502909e-07,3.397952,0.000009,-211.966954,141.673364,152.532897,10.859533,11322.328779,...,14175.616038,0.0,14175.616038,25497.944817,0.0,10.859533,25285.977872,168.868722,186.639931,94.669788
2030-01-01 01:00:00,135.08,62.380790,1.505150e-07,3.361920,0.000009,-209.719228,141.271696,152.250951,10.979255,11322.328779,...,14175.616038,0.0,14175.616038,25497.944817,0.0,10.979255,25288.225598,169.241244,187.209251,94.870195
2030-01-01 02:00:00,137.56,62.380822,1.508912e-07,2.643384,0.000009,-164.896444,142.511360,150.841220,8.329860,11322.328779,...,14175.616038,0.0,14175.616038,25497.944817,0.0,8.329860,25333.048382,170.554737,184.159991,95.437329
2030-01-01 03:00:00,138.00,62.380812,1.514246e-07,3.388815,0.000009,-211.397040,144.273570,155.634305,11.360735,11322.328779,...,14175.616038,0.0,14175.616038,25497.944817,0.0,11.360735,25286.547786,165.542535,183.235854,92.802997
2030-01-01 04:00:00,133.80,62.380793,1.520873e-07,5.477352,0.000009,-341.681537,143.653096,160.991283,17.338187,11322.328779,...,14175.616038,0.0,14175.616038,25497.944817,0.0,17.338187,25156.263289,160.624313,188.013926,90.512194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2030-12-31 19:00:00,129.64,61.965189,1.650526e-07,7.473269,0.000010,-463.082498,142.610081,162.682961,20.072879,11322.328779,...,14175.616038,0.0,14175.616038,25497.944817,0.0,20.072879,25034.862329,159.268884,193.110632,90.183621
2030-12-31 20:00:00,126.72,61.965189,1.589420e-07,7.392234,0.000010,-458.061188,139.545444,159.581552,20.036108,11322.328779,...,14175.616038,0.0,14175.616038,25497.944817,0.0,20.036108,25039.883639,162.440175,197.600092,91.960872
2030-12-31 21:00:00,122.52,61.965189,1.590419e-07,10.819507,0.000010,-670.432786,139.568569,162.682961,23.114391,11322.328779,...,14175.616038,0.0,14175.616038,25497.944817,0.0,23.114391,24827.512040,158.688996,202.640484,90.605707


In [ ]:
# Get average grid price (calculated once to avoid repetition)
grid_price = n.buses_t.marginal_price.filter(regex='^(?!.*C&I)').mean(axis=1)

# Initialize the DataFrame with all C&I related data
ci_data = pd.DataFrame(index=n.loads_t.p.index)

# Load
ci_data['C&I Load'] = n.loads_t.p.filter(regex='C&I').sum(axis=1)

# Import/Export volumes and costs
ci_data['C&I Import Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Import').sum(axis=1)
ci_data['C&I Export Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Export').sum(axis=1)
ci_data['C&I Import Cost'] = ci_data['C&I Import Volume'] * grid_price
ci_data['C&I Export Revenue'] = -ci_data['C&I Export Volume'] * grid_price  # Negative because exports generate revenue


# Group generators by technology type
ci_generators = n.generators[n.generators.index.str.contains('C&I')]
tech_groups = ci_generators.groupby('carrier')

# Generation and Curtailment by technology
for tech, group in tech_groups:
    tech_name = f'{tech}'

    # Generation
    ci_data[f'{tech_name} Generation'] = n.generators_t.p[group.index].sum(axis=1)

    # Potential Generation
    ci_data[f'{tech_name} Potential'] = (
        n.generators_t.p_max_pu[group.index] * 
        n.generators.p_nom_opt[group.index]
    ).sum(axis=1)

    # Curtailment
    ci_data[f'{tech_name} Curtailment'] = (
        ci_data[f'{tech_name} Potential'] - 
        ci_data[f'{tech_name} Generation']
    )

    # OPEX by technology
    ci_data[f'{tech_name} OPEX'] = (
        n.generators_t.p[group.index] * 
        group['marginal_cost'].values
    ).sum(axis=1)

    # Hourly expanded CAPEX by technology (using statistics.expanded_capex)
    df.loc[:, 'capex'] = df['p_nom_opt'] * df['capital_cost']
    capex_hourly = tech_expanded_capex / 8760
    ci_data[f'{tech_name} CAPEX'] = capex_hourly

    # Total cost by technology
    ci_data[f'{tech_name} Total Cost'] = (
        ci_data[f'{tech_name} CAPEX'] + 
        ci_data[f'{tech_name} OPEX']
    )

# # Calculate total system cost
# ci_data['C&I Total Cost'] = (
#     ci_data['C&I CAPEX'] + 
#     ci_data['C&I OPEX'] + 
#     ci_data['C&I Import Cost'] + 
#     ci_data['C&I Export Revenue']
# )

# # Calculate system-wide unit costs
# ci_data['Unit Cost (All Energy)'] = ci_data['C&I Total Cost']/(ci_data['C&I Load'] + ci_data['C&I Export Volume'] + ci_data['C&I Curtailment'])
# ci_data['Unit Cost (C&I Energy only)'] = ci_data['C&I Total Cost']/(ci_data['C&I Load'])

                                                               bus control  \
Generator                                                                    
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...  JPN08 C&I Grid      PQ   

                                                                       type  \
Generator                                                                     
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...  onshorewind-unspecified   

                                                    p_nom  p_nom_mod  \
Generator                                                              
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...    0.0        0.0   

                                                    p_nom_extendable  \
Generator                                                              
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...              True   

                                                    p_nom_min  p_nom_max  \
Generator        

In [ ]:
# Get average grid price (calculated once to avoid repetition)
grid_price = n.buses_t.marginal_price.filter(regex='^(?!.*C&I)').mean(axis=1)

# Initialize the DataFrame with all C&I related data
ci_data = pd.DataFrame(index=n.loads_t.p.index)

# Load
ci_data['C&I Load'] = n.loads_t.p.filter(regex='C&I').sum(axis=1)

# Import/Export volumes and costs
ci_data['C&I Import Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Import').sum(axis=1)
ci_data['C&I Export Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Export').sum(axis=1)
ci_data['C&I Import Cost'] = ci_data['C&I Import Volume'] * grid_price
ci_data['C&I Export Revenue'] = -ci_data['C&I Export Volume'] * grid_price  # Negative because exports generate revenue


# Group generators by technology type
ci_generators = n.generators[n.generators.index.str.contains('C&I')]
tech_groups = ci_generators.groupby('carrier')

# Generation and Curtailment by technology
for tech, group in tech_groups:
    tech_name = f'{tech}'

    # Generation
    ci_data[f'{tech_name} Generation'] = n.generators_t.p[group.index].sum(axis=1)

    # Potential Generation
    ci_data[f'{tech_name} Potential'] = (
        n.generators_t.p_max_pu[group.index] * 
        n.generators.p_nom_opt[group.index]
    ).sum(axis=1)

    # Curtailment
    ci_data[f'{tech_name} Curtailment'] = (
        ci_data[f'{tech_name} Potential'] - 
        ci_data[f'{tech_name} Generation']
    )

    # OPEX by technology
    ci_data[f'{tech_name} OPEX'] = (
        n.generators_t.p[group.index] * 
        group['marginal_cost'].values
    ).sum(axis=1)

    # Hourly expanded CAPEX by technology (using statistics.expanded_capex)
    df.loc[:, 'capex'] = df['p_nom_opt'] * df['capital_cost']
    capex_hourly = tech_expanded_capex / 8760
    ci_data[f'{tech_name} CAPEX'] = capex_hourly

    # Total cost by technology
    ci_data[f'{tech_name} Total Cost'] = (
        ci_data[f'{tech_name} CAPEX'] + 
        ci_data[f'{tech_name} OPEX']
    )

# # Calculate total system cost
# ci_data['C&I Total Cost'] = (
#     ci_data['C&I CAPEX'] + 
#     ci_data['C&I OPEX'] + 
#     ci_data['C&I Import Cost'] + 
#     ci_data['C&I Export Revenue']
# )

# # Calculate system-wide unit costs
# ci_data['Unit Cost (All Energy)'] = ci_data['C&I Total Cost']/(ci_data['C&I Load'] + ci_data['C&I Export Volume'] + ci_data['C&I Curtailment'])
# ci_data['Unit Cost (C&I Energy only)'] = ci_data['C&I Total Cost']/(ci_data['C&I Load'])

                                                               bus control  \
Generator                                                                    
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...  JPN08 C&I Grid      PQ   

                                                                       type  \
Generator                                                                     
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...  onshorewind-unspecified   

                                                    p_nom  p_nom_mod  \
Generator                                                              
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...    0.0        0.0   

                                                    p_nom_extendable  \
Generator                                                              
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...              True   

                                                    p_nom_min  p_nom_max  \
Generator        

In [ ]:
# Get average grid price (calculated once to avoid repetition)
grid_price = n.buses_t.marginal_price.filter(regex='^(?!.*C&I)').mean(axis=1)

# Initialize the DataFrame with all C&I related data
ci_data = pd.DataFrame(index=n.loads_t.p.index)

# Load
ci_data['C&I Load'] = n.loads_t.p.filter(regex='C&I').sum(axis=1)

# Import/Export volumes and costs
ci_data['C&I Import Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Import').sum(axis=1)
ci_data['C&I Export Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Export').sum(axis=1)
ci_data['C&I Import Cost'] = ci_data['C&I Import Volume'] * grid_price
ci_data['C&I Export Revenue'] = -ci_data['C&I Export Volume'] * grid_price  # Negative because exports generate revenue


# Group generators by technology type
ci_generators = n.generators[n.generators.index.str.contains('C&I')]
tech_groups = ci_generators.groupby('carrier')

# Generation and Curtailment by technology
for tech, group in tech_groups:
    tech_name = f'{tech}'

    # Generation
    ci_data[f'{tech_name} Generation'] = n.generators_t.p[group.index].sum(axis=1)

    # Potential Generation
    ci_data[f'{tech_name} Potential'] = (
        n.generators_t.p_max_pu[group.index] * 
        n.generators.p_nom_opt[group.index]
    ).sum(axis=1)

    # Curtailment
    ci_data[f'{tech_name} Curtailment'] = (
        ci_data[f'{tech_name} Potential'] - 
        ci_data[f'{tech_name} Generation']
    )

    # OPEX by technology
    ci_data[f'{tech_name} OPEX'] = (
        n.generators_t.p[group.index] * 
        group['marginal_cost'].values
    ).sum(axis=1)

    # Hourly expanded CAPEX by technology (using statistics.expanded_capex)
    df.loc[:, 'capex'] = df['p_nom_opt'] * df['capital_cost']
    capex_hourly = tech_expanded_capex / 8760
    ci_data[f'{tech_name} CAPEX'] = capex_hourly

    # Total cost by technology
    ci_data[f'{tech_name} Total Cost'] = (
        ci_data[f'{tech_name} CAPEX'] + 
        ci_data[f'{tech_name} OPEX']
    )

# # Calculate total system cost
# ci_data['C&I Total Cost'] = (
#     ci_data['C&I CAPEX'] + 
#     ci_data['C&I OPEX'] + 
#     ci_data['C&I Import Cost'] + 
#     ci_data['C&I Export Revenue']
# )

# # Calculate system-wide unit costs
# ci_data['Unit Cost (All Energy)'] = ci_data['C&I Total Cost']/(ci_data['C&I Load'] + ci_data['C&I Export Volume'] + ci_data['C&I Curtailment'])
# ci_data['Unit Cost (C&I Energy only)'] = ci_data['C&I Total Cost']/(ci_data['C&I Load'])

                                                               bus control  \
Generator                                                                    
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...  JPN08 C&I Grid      PQ   

                                                                       type  \
Generator                                                                     
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...  onshorewind-unspecified   

                                                    p_nom  p_nom_mod  \
Generator                                                              
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...    0.0        0.0   

                                                    p_nom_extendable  \
Generator                                                              
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...              True   

                                                    p_nom_min  p_nom_max  \
Generator        

In [ ]:
# Get average grid price (calculated once to avoid repetition)
grid_price = n.buses_t.marginal_price.filter(regex='^(?!.*C&I)').mean(axis=1)

# Initialize the DataFrame with all C&I related data
ci_data = pd.DataFrame(index=n.loads_t.p.index)

# Load
ci_data['C&I Load'] = n.loads_t.p.filter(regex='C&I').sum(axis=1)

# Import/Export volumes and costs
ci_data['C&I Import Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Import').sum(axis=1)
ci_data['C&I Export Volume'] = n.links_t.p0.filter(regex='C&I').filter(regex='Export').sum(axis=1)
ci_data['C&I Import Cost'] = ci_data['C&I Import Volume'] * grid_price
ci_data['C&I Export Revenue'] = -ci_data['C&I Export Volume'] * grid_price  # Negative because exports generate revenue


# Group generators by technology type
ci_generators = n.generators[n.generators.index.str.contains('C&I')]
tech_groups = ci_generators.groupby('carrier')

# Generation and Curtailment by technology
for tech, group in tech_groups:
    tech_name = f'{tech}'

    # Generation
    ci_data[f'{tech_name} Generation'] = n.generators_t.p[group.index].sum(axis=1)

    # Potential Generation
    ci_data[f'{tech_name} Potential'] = (
        n.generators_t.p_max_pu[group.index] * 
        n.generators.p_nom_opt[group.index]
    ).sum(axis=1)

    # Curtailment
    ci_data[f'{tech_name} Curtailment'] = (
        ci_data[f'{tech_name} Potential'] - 
        ci_data[f'{tech_name} Generation']
    )

    # OPEX by technology
    ci_data[f'{tech_name} OPEX'] = (
        n.generators_t.p[group.index] * 
        group['marginal_cost'].values
    ).sum(axis=1)

    # Hourly expanded CAPEX by technology (using statistics.expanded_capex)
    df.loc[:, 'capex'] = df['p_nom_opt'] * df['capital_cost']
    capex_hourly = tech_expanded_capex / 8760
    ci_data[f'{tech_name} CAPEX'] = capex_hourly

    # Total cost by technology
    ci_data[f'{tech_name} Total Cost'] = (
        ci_data[f'{tech_name} CAPEX'] + 
        ci_data[f'{tech_name} OPEX']
    )

# # Calculate total system cost
# ci_data['C&I Total Cost'] = (
#     ci_data['C&I CAPEX'] + 
#     ci_data['C&I OPEX'] + 
#     ci_data['C&I Import Cost'] + 
#     ci_data['C&I Export Revenue']
# )

# # Calculate system-wide unit costs
# ci_data['Unit Cost (All Energy)'] = ci_data['C&I Total Cost']/(ci_data['C&I Load'] + ci_data['C&I Export Volume'] + ci_data['C&I Curtailment'])
# ci_data['Unit Cost (C&I Energy only)'] = ci_data['C&I Total Cost']/(ci_data['C&I Load'])

                                                               bus control  \
Generator                                                                    
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...  JPN08 C&I Grid      PQ   

                                                                       type  \
Generator                                                                     
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...  onshorewind-unspecified   

                                                    p_nom  p_nom_mod  \
Generator                                                              
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...    0.0        0.0   

                                                    p_nom_extendable  \
Generator                                                              
JPN08 C&I Grid-onshorewind-unspecified-ext-2030...              True   

                                                    p_nom_min  p_nom_max  \
Generator        

In [143]:
ci_data

,C&I Load,C&I Import Volume,C&I Export Volume,C&I Import Cost,C&I Export Revenue,OnshoreWind Generation,OnshoreWind Potential,OnshoreWind Curtailment,OnshoreWind OPEX,OnshoreWind CAPEX,OnshoreWind Total Cost,SolarGrid Generation,SolarGrid Potential,SolarGrid Curtailment,SolarGrid OPEX,SolarGrid CAPEX,SolarGrid Total Cost
snapshot,,,,,,,,,,,,,,,,,
2030-01-01 00:00:00,135.48,1.502909e-07,3.397952,0.000009,-211.966954,141.673364,152.532897,10.859533,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2030-01-01 01:00:00,135.08,1.505150e-07,3.361920,0.000009,-209.719228,141.271696,152.250951,10.979255,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2030-01-01 02:00:00,137.56,1.508912e-07,2.643384,0.000009,-164.896444,142.511360,150.841220,8.329860,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2030-01-01 03:00:00,138.00,1.514246e-07,3.388815,0.000009,-211.397040,144.273570,155.634305,11.360735,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2030-01-01 04:00:00,133.80,1.520873e-07,5.477352,0.000009,-341.681537,143.653096,160.991283,17.338187,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2030-12-31 19:00:00,129.64,1.650526e-07,7.473269,0.000010,-463.082498,142.610081,162.682961,20.072879,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2030-12-31 20:00:00,126.72,1.589420e-07,7.392234,0.000010,-458.061188,139.545444,159.581552,20.036108,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2030-12-31 21:00:00,122.52,1.590419e-07,10.819507,0.000010,-670.432786,139.568569,162.682961,23.114391,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [107]:
ci_data.to_csv('ci_data.csv')